# Streaming Cache Sweep on GSM8K

Streaming refine：pruned warm-up 后，suffix tokens 移到 input 侧（而非留在 KV cache 中），
每步 refine 时 suffix 参与双向注意力，模型能正确感知后续空间。

两组实验 × window_blocks 1-3 = **6 个任务**：

| Group | 说明 |
|-------|------|
| **plain** | 原版 dual-cache（无 expand）→ streaming refine |
| **expand** | dual-cache + expand（mtr=0.7, rewarm+fb）→ streaming refine |

seed=42 锁定，与 sweep_suffix_prune_gsm8k 完全可比。

## 1. 环境设置

In [ ]:
import os
import torch
import gc

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} '
          f'({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)')

## 2. 配置 & 任务池 / GPU 池

In [ ]:
import subprocess
import datetime
import threading
import queue

SEED = 42

# ========== Sweep axis ==========
WINDOW_BLOCKS_RANGE = [1, 2, 3]

TASK_CONFIGS = []

# --- Group "plain": streaming refine, no expand ---
for wb in WINDOW_BLOCKS_RANGE:
    TASK_CONFIGS.append({
        'group': 'plain', 'mode': 'streaming', 'window_blocks': wb,
        'streaming_cache': True,
        'mid_block_expand': False,
    })

# --- Group "expand": streaming refine + expand (mtr=0.7, rewarm+fb) ---
for wb in WINDOW_BLOCKS_RANGE:
    TASK_CONFIGS.append({
        'group': 'expand', 'mode': 'streaming_expand', 'window_blocks': wb,
        'streaming_cache': True,
        'mid_block_expand': True,
        'mid_trigger_ratio': 0.7,
        'rewarm_on_expand': True,
        'front_block_fallback_only': True,
    })


def config_name(cfg):
    return f"{cfg['group']}_sw{cfg['window_blocks']}"

ALL_GROUPS = sorted(set(c['group'] for c in TASK_CONFIGS))


# ========== GPU pool ==========
GPU_POOL = [int(x) for x in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]

# ========== Eval params ==========
task = 'gsm8k'
fewshot = 5
limit = None        # full GSM8K (1319 samples)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

gen_length = 256
steps = 256
block_length = 32
threshold = 0.9

print(f'Total tasks: {len(TASK_CONFIGS)}  (seed={SEED})')
for cfg in TASK_CONFIGS:
    print(f'  {config_name(cfg):24s}  group={cfg["group"]:8s}  wb={cfg["window_blocks"]}  expand={cfg["mid_block_expand"]}')
print(f'GPU pool: {GPU_POOL} ({len(GPU_POOL)} GPUs)')
print(f'limit: {limit} ({"full" if limit is None else f"{limit} samples"})')
print(f'Timestamp: {timestamp}')

## 3. 启动任务池

In [ ]:
task_queue = queue.Queue()
for cfg in TASK_CONFIGS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_results = []


def gpu_worker(gpu_id):
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = config_name(cfg)
        log_file = f'nlogs/streaming_{task}_{name}_{timestamp}.log'
        output_dir = f'evals_results/streaming_sweep/{task}-{name}-{timestamp}'

        base_args = [
            f"model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f'gen_length={gen_length}',
            f'steps={steps}',
            f'block_length={block_length}',
            f'threshold={threshold}',
            'use_cache=True',
            'show_speed=True',
            f'seed={SEED}',
        ]

        extra_args = [
            'streaming_cache=True',
            f"window_blocks={cfg['window_blocks']}",
        ]
        if cfg['mid_block_expand']:
            extra_args += [
                'dual_cache=True',
                'mid_block_expand=True',
                f"mid_trigger_ratio={cfg['mid_trigger_ratio']}",
                f"rewarm_on_expand={cfg['rewarm_on_expand']}",
                f"front_block_fallback_only={cfg['front_block_fallback_only']}",
            ]

        model_args_str = ','.join(base_args + extra_args)

        limit_str = f' --limit {limit}' if limit is not None else ''
        cmd = (
            f'CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py '
            f'--tasks {task} --num_fewshot {fewshot}{limit_str} '
            f'--confirm_run_unsafe_code --model llada_dist '
            f'--model_args {model_args_str} '
            f'--output_path {output_dir} --log_samples'
        )

        print(f'[GPU {gpu_id}] START  {name}')

        p = subprocess.Popen(
            cmd, shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()

        status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
        print(f'[GPU {gpu_id}] DONE   {name}  {status}')

        with results_lock:
            all_results.append((cfg, name, log_file, output_dir, rc))

        task_queue.task_done()


threads = []
for gpu_id in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gpu_id,), daemon=True)
    t.start()
    threads.append(t)

print(f'\nLaunched {len(threads)} GPU workers for {len(TASK_CONFIGS)} tasks.')
print('Waiting for all tasks to complete...')

In [ ]:
for t in threads:
    t.join()

print(f'\nAll {len(all_results)} / {len(TASK_CONFIGS)} tasks finished.')
for cfg, name, log_file, output_dir, rc in all_results:
    status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
    print(f'  {name:24s}  {status}  log={log_file}')

## 4. 解析评测结果

In [ ]:
import re
import json
import pandas as pd

parsed_results = []
for cfg in TASK_CONFIGS:
    name = config_name(cfg)
    log_file = f'nlogs/streaming_{task}_{name}_{timestamp}.log'
    if not os.path.exists(log_file):
        print(f'WARNING: {log_file} not found')
        continue
    with open(log_file, 'r') as f:
        content = f.read()

    acc_match = re.search(r'exact_match.*?[\|,]\s*[\|]?\s*([\d.]+)', content)
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    token_match = re.search(r'Total number of tokens generated:\s*(\d+)', content)

    parsed_results.append({
        'group': cfg['group'],
        'mode': cfg['mode'],
        'window_blocks': cfg['window_blocks'],
        'config': name,
        'accuracy': float(acc_match.group(1)) if acc_match else None,
        'tokens_per_sec': float(speed_match.group(1)) if speed_match else None,
        'total_nfe': int(nfe_match.group(1)) if nfe_match else None,
        'time_sec': float(time_match.group(1)) if time_match else None,
        'total_tokens': int(token_match.group(1)) if token_match else None,
        'log_file': log_file,
    })

df = pd.DataFrame(parsed_results)
show_cols = ['config', 'group', 'accuracy', 'total_nfe', 'time_sec', 'total_tokens', 'tokens_per_sec']
display(df[show_cols].reset_index(drop=True))

## 5. Accuracy / NFE / Speed 对比图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

GROUP_COLORS = {'expand': '#FF5722', 'plain': '#2196F3'}
METRICS = [
    ('accuracy', 'GSM8K Accuracy', 'Accuracy'),
    ('total_nfe', 'Total NFE', 'NFE'),
    ('tokens_per_sec', 'Tokens / sec', 'Speed'),
]

df_valid = df.dropna(subset=['accuracy']).copy()
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for group_name, color in GROUP_COLORS.items():
    sub = df_valid[df_valid['group'] == group_name].sort_values('window_blocks')
    if sub.empty:
        continue
    wb = sub['window_blocks'].values

    for ax_idx, (col, ylabel, title) in enumerate(METRICS):
        vals = sub[col].values
        if pd.isna(vals).all():
            continue
        axes[ax_idx].plot(wb, vals, 'o-', color=color,
                          label=group_name, linewidth=2, markersize=8)
        for w, v in zip(wb, vals):
            if pd.notna(v):
                fmt = f'{v:.4f}' if col == 'accuracy' else (str(int(v)) if col == 'total_nfe' else f'{v:.1f}')
                axes[ax_idx].annotate(fmt, (w, v), textcoords='offset points',
                                      xytext=(0, 10), ha='center', fontsize=8, color=color)

for ax_idx, (col, ylabel, title) in enumerate(METRICS):
    axes[ax_idx].set_xlabel('window_blocks')
    axes[ax_idx].set_xticks(WINDOW_BLOCKS_RANGE)
    axes[ax_idx].set_ylabel(ylabel)
    axes[ax_idx].set_title(f'{title} vs window_blocks', fontweight='bold')
    axes[ax_idx].legend(fontsize=8)
    axes[ax_idx].grid(True, alpha=0.3)

fig.suptitle('Streaming Refine Sweep (GSM8K, seed=42)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../eval_results', exist_ok=True)
plt.savefig('../eval_results/streaming_cache_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval_results/streaming_cache_sweep.png')

## 6. 从已有结果重新加载

In [ ]:
import glob, re, json, os
import pandas as pd

task = 'gsm8k'
SEED = 42
WINDOW_BLOCKS_RANGE = [1, 2, 3]

TASK_CONFIGS = []
for wb in WINDOW_BLOCKS_RANGE:
    TASK_CONFIGS.append({'group': 'plain', 'mode': 'streaming', 'window_blocks': wb,
                         'streaming_cache': True, 'mid_block_expand': False})
for wb in WINDOW_BLOCKS_RANGE:
    TASK_CONFIGS.append({'group': 'expand', 'mode': 'streaming_expand', 'window_blocks': wb,
                         'streaming_cache': True, 'mid_block_expand': True,
                         'mid_trigger_ratio': 0.7, 'rewarm_on_expand': True,
                         'front_block_fallback_only': True})
ALL_GROUPS = sorted(set(c['group'] for c in TASK_CONFIGS))

def config_name(cfg):
    return f"{cfg['group']}_sw{cfg['window_blocks']}"

first_name = config_name(TASK_CONFIGS[0])
latest_logs = sorted(
    glob.glob(f'nlogs/streaming_{task}_{first_name}_*.log'),
    key=os.path.getmtime, reverse=True,
)
if latest_logs:
    fname = os.path.basename(latest_logs[0])
    timestamp = fname.replace(f'streaming_{task}_{first_name}_', '').replace('.log', '')
    print(f'Auto-detected latest timestamp: {timestamp}')
else:
    timestamp = 'NOTFOUND'
    print('WARNING: No log found!')

parsed_results = []
for cfg in TASK_CONFIGS:
    name = config_name(cfg)
    log_file = f'nlogs/streaming_{task}_{name}_{timestamp}.log'
    if not os.path.exists(log_file):
        continue
    with open(log_file, 'r') as f:
        content = f.read()

    acc_match = re.search(r'exact_match.*?[\|,]\s*[\|]?\s*([\d.]+)', content)
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    token_match = re.search(r'Total number of tokens generated:\s*(\d+)', content)
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)

    parsed_results.append({
        'group': cfg['group'],
        'mode': cfg['mode'],
        'window_blocks': cfg['window_blocks'],
        'config': name,
        'accuracy': float(acc_match.group(1)) if acc_match else None,
        'total_nfe': int(nfe_match.group(1)) if nfe_match else None,
        'time_sec': float(time_match.group(1)) if time_match else None,
        'total_tokens': int(token_match.group(1)) if token_match else None,
        'tokens_per_sec': float(speed_match.group(1)) if speed_match else None,
    })

df = pd.DataFrame(parsed_results)
print(f'Loaded {len(parsed_results)} parsed_results  (seed={SEED})\n')

header = '\t'.join(['Config', 'Group', 'Mode', 'WB', 'ACC', 'NFE', 'Time(s)', 'Total Token', 'Token/s'])
print(header)
for r in parsed_results:
    row = '\t'.join([
        r['config'],
        r['group'],
        r['mode'],
        str(r['window_blocks']),
        f"{r['accuracy']:.4f}" if r['accuracy'] is not None else '',
        str(r['total_nfe']) if r['total_nfe'] is not None else '',
        f"{r['time_sec']:.1f}" if r['time_sec'] is not None else '',
        str(r['total_tokens']) if r['total_tokens'] is not None else '',
        f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] is not None else '',
    ])
    print(row)

print(f'\nRe-run Section 5 to regenerate plots.')